# Example: Multi-Class Classification with Cross-Entropy Loss in PyTorch

## Learning goals

This notebook demonstrates a very small **3-class neural-network classifier** using PyTorch.

You will learn:

- how to create a tiny training dataset and a separate hypothetical validation dataset;
- how to define a neural network with **3 output neurons**;
- what **logits** are;
- why `forward()` should return **raw logits**;
- how `nn.CrossEntropyLoss()` uses those logits directly;
- why Softmax should **not** be placed inside `forward()` when using `CrossEntropyLoss`;
- how to identify the predicted class from the logits;
- how to calculate **loss and accuracy on both the training and validation sets**;
- why the validation set is evaluated without updating the network;
- how backpropagation and SGD update the network.

> **Main rule:** When using `nn.CrossEntropyLoss()`, let `forward()` return raw logits. Do not apply `torch.softmax()` inside `forward()`.


In [2]:
import sys
print(sys.executable)

/home/neo/Code/ML/.venv/bin/python


## 1. Import PyTorch

- `torch` provides tensors and basic PyTorch operations.
- `torch.nn` contains neural-network layers and loss functions.
- `torch.optim` contains optimisation algorithms such as SGD.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim


## 2. Create a tiny hypothetical dataset

We will use only **4 training examples**.

Each example has **2 input features**.


In [5]:
# There are 4 training examples.
# Each example has 2 input features.

X = torch.tensor([
    [1.0, 2.0],   # Example 1
    [2.0, 1.0],   # Example 2
    [3.0, 3.0],   # Example 3
    [4.0, 1.0]    # Example 4
])

print("Input data X:")
print(X)

print("\nShape of X:")
print(X.shape)


Input data X:
tensor([[1., 2.],
        [2., 1.],
        [3., 3.],
        [4., 1.]])

Shape of X:
torch.Size([4, 2])


`X.shape` should be:

```text
torch.Size([4, 2])
```

This means:

- 4 rows = 4 examples;
- 2 columns = 2 features per example.


## 3. Create the correct class labels

There are **3 possible classes**:

- Class 0
- Class 1
- Class 2

For `CrossEntropyLoss`, each correct answer is represented by its **class number**.


In [6]:
Y = torch.tensor([
    0,   # Example 1 belongs to Class 0
    1,   # Example 2 belongs to Class 1
    2,   # Example 3 belongs to Class 2
    1    # Example 4 belongs to Class 1
])

print("Correct class labels:")
print(Y)


Correct class labels:
tensor([0, 1, 2, 1])


The labels mean:

| Example | Correct class |
|---|---:|
| 1 | 0 |
| 2 | 1 |
| 3 | 2 |
| 4 | 1 |

We do **not** need to represent the labels as probability vectors such as `[1, 0, 0]`. `CrossEntropyLoss` can use the integer class labels directly.


## 4. Create a small hypothetical validation set

A **validation set** contains examples that are kept separate from the training set.

The network does **not** use these validation examples to update its weights and biases.

Instead, we use the validation set to check how well the network performs on examples that were not used for learning.

For this teaching example, we will create 3 hypothetical validation examples. Each example still has 2 input features, and the class labels are still 0, 1, or 2.


In [8]:
# Hypothetical validation input examples.
X_validation = torch.tensor([
    [1.2, 1.8],   # Validation example 1
    [2.8, 3.2],   # Validation example 2
    [3.8, 1.2]    # Validation example 3
])

# Correct class labels for the validation examples.
Y_validation = torch.tensor([
    0,   # Validation example 1 belongs to Class 0
    2,   # Validation example 2 belongs to Class 2
    1    # Validation example 3 belongs to Class 1
])

print("Validation input data:")
print(X_validation)

print("\nValidation labels:")
print(Y_validation)

print("\nShape of X_validation:")
print(X_validation.shape)


Validation input data:
tensor([[1.2000, 1.8000],
        [2.8000, 3.2000],
        [3.8000, 1.2000]])

Validation labels:
tensor([0, 2, 1])

Shape of X_validation:
torch.Size([3, 2])


The validation set has shape:

```text
torch.Size([3, 2])
```

This means:

- 3 validation examples;
- 2 input features per example.

The important distinction is:

```text
Training set
    ↓
used to calculate gradients
    ↓
used to update weights and biases

Validation set
    ↓
NOT used to calculate training updates
    ↓
used only to measure performance
```


## 5. Define the neural network

The architecture is:

```text
2 input features
      ↓
4 hidden neurons
      ↓
ReLU
      ↓
3 output neurons
```

We need **3 output neurons because there are 3 possible classes**.

Each output neuron corresponds to one class:

```text
Output neuron 0 → Class 0
Output neuron 1 → Class 1
Output neuron 2 → Class 2
```


In [9]:
class SimpleClassifier(nn.Module):

    def __init__(self):
        # Initialise the parent PyTorch neural-network class.
        super().__init__()

        # Hidden layer:
        # 2 input features -> 4 hidden neurons.
        self.hidden_layer = nn.Linear(2, 4)

        # Output layer:
        # 4 hidden neurons -> 3 output neurons.
        self.output_layer = nn.Linear(4, 3)

    def forward(self, x):
        # Send the input through the hidden layer.
        x = self.hidden_layer(x)

        # Apply the ReLU activation function.
        x = torch.relu(x)

        # Produce 3 RAW output values for each example.
        # These raw values are called LOGITS.
        logits = self.output_layer(x)

        # IMPORTANT:
        # Do NOT apply torch.softmax() here.
        # CrossEntropyLoss expects the raw logits.
        return logits


### What are logits?

A **logit** is a raw score from an output neuron.

For one example, the network might produce:

```text
[1.8, 0.4, -0.7]
```

These are **not probabilities**. They do not need to add to 1, and they may even be negative.

The largest logit indicates the class currently preferred by the model.


## 6. Create the model

Defining the class describes the network. The following line creates an actual network object.


In [10]:
model = SimpleClassifier()

print(model)


SimpleClassifier(
  (hidden_layer): Linear(in_features=2, out_features=4, bias=True)
  (output_layer): Linear(in_features=4, out_features=3, bias=True)
)


## 7. Define the cross-entropy loss function

`nn.CrossEntropyLoss()` is commonly used for **multi-class classification**.

Most importantly, it expects **raw logits** from the model.


In [11]:
loss_function = nn.CrossEntropyLoss()


### Why is Softmax not inside `forward()`?

Conceptually, we can think of classification as:

```text
Logits
  ↓
Softmax
  ↓
Probabilities
  ↓
Cross-entropy
```

However, PyTorch's `CrossEntropyLoss` performs the required Softmax-related calculation internally in a numerically stable way.

Therefore, during training we use:

```python
logits = model(X)
loss = loss_function(logits, Y)
```

We should **not** do this:

```python
probabilities = torch.softmax(logits, dim=1)
loss = loss_function(probabilities, Y)
```


## 8. Define the optimiser

The optimiser updates the model's trainable weights and biases.

Here we use **SGD** (Stochastic Gradient Descent) with a learning rate of `0.1`.


In [12]:
optimizer = optim.SGD(
    model.parameters(),  # All trainable weights and biases.
    lr=0.1               # Learning rate.
)


## 9. Look at the raw logits before training

Calling `model(X)` automatically calls the model's `forward()` method.

Because there are 4 examples and 3 classes, the output has shape **4 × 3**.


In [13]:
logits = model(X)

print("Raw logits before training:")
print(logits)

print("\nShape of logits:")
print(logits.shape)


Raw logits before training:
tensor([[-0.3126,  0.1799, -0.0469],
        [-0.2789,  0.2987, -0.0836],
        [-0.3900, -0.0935,  0.0378],
        [-0.2915,  0.2542, -0.0699]], grad_fn=<AddmmBackward0>)

Shape of logits:
torch.Size([4, 3])


Each row belongs to one example, and each column belongs to one class:

```text
             Class 0   Class 1   Class 2
Example 1      ...       ...       ...
Example 2      ...       ...       ...
Example 3      ...       ...       ...
Example 4      ...       ...       ...
```

The exact values will vary because the model begins with randomly initialised weights and biases.


## 10. Calculate cross-entropy loss directly from the logits

This is the key step:

```python
loss = loss_function(logits, Y)
```

The loss function receives:

1. the **raw logits**;
2. the **correct class labels**.

No explicit Softmax is required.


In [10]:
loss = loss_function(logits, Y)

print("Cross-entropy loss:")
print(loss.item())


Cross-entropy loss:
0.959670901298523


The loss is one number measuring how well the model's outputs agree with the correct classes.

In general:

- smaller loss = better;
- larger loss = worse.


## 11. Train the neural network and evaluate both datasets

An **epoch** is one complete pass through the training dataset.

During each epoch, we will do two different jobs.

### Part A — Training

1. put the model into training mode;
2. clear old gradients;
3. perform forward propagation on the **training set**;
4. calculate the training cross-entropy loss;
5. perform backpropagation;
6. update the weights and biases.

### Part B — Evaluation

After the parameter update, we will evaluate the current network on:

- the **training set**; and
- the **validation set**.

For evaluation, we use:

```python
model.eval()
```

and:

```python
with torch.no_grad():
```

`model.eval()` puts the network into evaluation mode.

`torch.no_grad()` tells PyTorch that gradients are not required. This saves memory and computation because evaluation does not perform backpropagation.

For both datasets, accuracy is calculated in the same way:

- For each example, choose the class having the largest output score.
- Check whether each predicted class is equal to the correct class.
- Convert correct answers to 1 and incorrect answers to 0, then calculate their average to obtain the accuracy.

```python
predictions = logits.argmax(dim=1)
correct_predictions = predictions == correct_labels
accuracy = correct_predictions.float().mean()
```

The validation set is **never passed to `loss.backward()`** and therefore does not directly change the weights or biases.


In [ ]:
number_of_epochs = 100

for epoch in range(number_of_epochs):

    # =====================================================
    # PART A: TRAIN THE NETWORK
    # =====================================================

    # Put the network into training mode.
    model.train()

    # Clear gradients left over from the previous epoch.
    optimizer.zero_grad()

    # -----------------------------------------------------
    # FORWARD PROPAGATION ON THE TRAINING SET
    # -----------------------------------------------------

    # Produce raw logits for the training examples.
    training_logits_for_backprop = model(X)

    # -----------------------------------------------------
    # TRAINING LOSS USED FOR BACKPROPAGATION
    # -----------------------------------------------------

    # CrossEntropyLoss receives raw logits and correct labels.
    training_loss_for_backprop = loss_function(
        training_logits_for_backprop,
        Y
    )

    # -----------------------------------------------------
    # BACKPROPAGATION
    # -----------------------------------------------------

    # Calculate gradients of the training loss with respect
    # to every trainable weight and bias.
    training_loss_for_backprop.backward()

    # -----------------------------------------------------
    # UPDATE THE PARAMETERS
    # -----------------------------------------------------

    # Use the gradients to update the weights and biases.
    optimizer.step() # weights_new = weights_old - learning_rate * gradient (Stochastic Gradient Descent)
    # the network now has new weights

    # =====================================================
    # PART B: EVALUATE THE CURRENT NETWORK
    # =====================================================

    # Put the network into evaluation mode.
    model.eval()

    # We do not need gradient calculations during evaluation.
    with torch.no_grad():

        # -------------------------------------------------
        # EVALUATE THE TRAINING SET
        # -------------------------------------------------

        # Run the UPDATED model on all training examples.
        training_logits = model(X)

        # Calculate training loss.
        training_loss = loss_function(training_logits, Y)

        # Find the predicted training class for each example.
        training_predictions = training_logits.argmax(dim=1)

        # Compare predictions with the correct training labels.
        training_correct = training_predictions == Y

        # Calculate the fraction of correct training predictions.
        training_accuracy = training_correct.float().mean()

        # -------------------------------------------------
        # EVALUATE THE VALIDATION SET
        # -------------------------------------------------

        # Run the same UPDATED model on the validation examples.
        validation_logits = model(X_validation)

        # Calculate validation loss.
        validation_loss = loss_function(
            validation_logits,
            Y_validation
        )

        # Find the predicted validation class for each example.
        validation_predictions = validation_logits.argmax(dim=1)

        # Compare predictions with the correct validation labels.
        validation_correct = validation_predictions == Y_validation

        # Calculate the fraction of correct validation predictions.
        validation_accuracy = validation_correct.float().mean()

    # =====================================================
    # DISPLAY THE RESULTS FOR THIS EPOCH
    # =====================================================

    print("Epoch:", epoch + 1)

    print(
        "Training loss:",
        round(training_loss.item(), 4),
        "| Training accuracy:",
        round(training_accuracy.item() * 100, 2),
        "%"
    )

    print(
        "Validation loss:",
        round(validation_loss.item(), 4),
        "| Validation accuracy:",
        round(validation_accuracy.item() * 100, 2),
        "%"
    )

    print()


Epoch: 1
Training loss: 0.932 | Training accuracy: 75.0 %
Validation loss: 0.9602 | Validation accuracy: 66.67 %

Epoch: 2
Training loss: 0.9107 | Training accuracy: 75.0 %
Validation loss: 0.944 | Validation accuracy: 66.67 %

Epoch: 3
Training loss: 0.8926 | Training accuracy: 75.0 %
Validation loss: 0.9299 | Validation accuracy: 66.67 %

Epoch: 4
Training loss: 0.8761 | Training accuracy: 75.0 %
Validation loss: 0.9172 | Validation accuracy: 66.67 %

Epoch: 5
Training loss: 0.8604 | Training accuracy: 75.0 %
Validation loss: 0.9053 | Validation accuracy: 66.67 %

Epoch: 6
Training loss: 0.845 | Training accuracy: 75.0 %
Validation loss: 0.8935 | Validation accuracy: 66.67 %

Epoch: 7
Training loss: 0.8321 | Training accuracy: 75.0 %
Validation loss: 0.8816 | Validation accuracy: 66.67 %

Epoch: 8
Training loss: 0.8166 | Training accuracy: 75.0 %
Validation loss: 0.8686 | Validation accuracy: 66.67 %

Epoch: 9
Training loss: 0.8011 | Training accuracy: 75.0 %
Validation loss: 0.8566 

## 12. What happens during one epoch?

```text
                 TRAINING PHASE
                 --------------

Training examples X
        ↓
model(X)
        ↓
training logits
        ↓
CrossEntropyLoss(training logits, Y)
        ↓
training loss
        ↓
loss.backward()
        ↓
gradients
        ↓
optimizer.step()
        ↓
weights and biases updated


                 EVALUATION PHASE
                 ----------------

             Updated model
                 /     \
                /       \
               ↓         ↓
        Training set   Validation set
             X          X_validation
               ↓         ↓
             logits     logits
               ↓         ↓
              loss       loss
               ↓         ↓
            accuracy   accuracy
```

The two datasets have different purposes:

- **Training loss and training accuracy** tell us how well the model currently performs on examples it has learned from.
- **Validation loss and validation accuracy** tell us how well the same model performs on separate examples that were not used to update its parameters.

A model can sometimes continue improving on the training set while its validation performance stops improving or becomes worse. That pattern is one sign of **overfitting**.

In this tiny hypothetical example there are very few samples, so the numerical results should be treated only as a teaching demonstration.


## Summary: training and validation measurements

The network learns **only from the training loss**:

```python
model.train()

optimizer.zero_grad()

training_logits_for_backprop = model(X)

training_loss_for_backprop = loss_function(
    training_logits_for_backprop,
    Y
)

training_loss_for_backprop.backward()

optimizer.step()
```

After the update, we measure performance without calculating gradients:

```python
model.eval()

with torch.no_grad():

    training_logits = model(X)
    training_loss = loss_function(training_logits, Y)
    training_accuracy = (
        training_logits.argmax(dim=1) == Y
    ).float().mean()

    validation_logits = model(X_validation)
    validation_loss = loss_function(
        validation_logits,
        Y_validation
    )
    validation_accuracy = (
        validation_logits.argmax(dim=1) == Y_validation
    ).float().mean()
```

The most important idea is:

> **Training data updates the model. Validation data evaluates the model but does not update it.**

Softmax is still not required for calculating either the loss or the predicted class.


## 13. Optional: use Softmax only to display probabilities

Softmax is still useful when we want to **interpret** the output as probabilities.

The distinction is:

- **For training:** use raw logits with `CrossEntropyLoss`.
- **For display:** apply `torch.softmax()` separately.


In [12]:
# Put the model into evaluation mode.
model.eval()

# We do not need gradients when merely inspecting predictions.
with torch.no_grad():

    # -----------------------------
    # Training set
    # -----------------------------
    training_logits = model(X)

    # Convert logits to probabilities ONLY for interpretation.
    training_probabilities = torch.softmax(
        training_logits,
        dim=1
    )

    training_predictions = training_logits.argmax(dim=1)

    # -----------------------------
    # Validation set
    # -----------------------------
    validation_logits = model(X_validation)

    validation_probabilities = torch.softmax(
        validation_logits,
        dim=1
    )

    validation_predictions = validation_logits.argmax(dim=1)

print("TRAINING SET")
print("Raw logits:")
print(training_logits)

print("\nProbabilities:")
print(training_probabilities)

print("\nPredicted classes:")
print(training_predictions)

print("\nCorrect classes:")
print(Y)

print("\n------------------------------\n")

print("VALIDATION SET")
print("Raw logits:")
print(validation_logits)

print("\nProbabilities:")
print(validation_probabilities)

print("\nPredicted classes:")
print(validation_predictions)

print("\nCorrect classes:")
print(Y_validation)


TRAINING SET
Raw logits:
tensor([[ 1.4384, -1.7723,  0.4567],
        [-2.4329,  1.3900, -1.0346],
        [-1.1421, -2.0920,  0.8119],
        [-6.7520,  3.2796, -1.7879]])

Probabilities:
tensor([[7.0673e-01, 2.8501e-02, 2.6477e-01],
        [1.9691e-02, 9.0060e-01, 7.9713e-02],
        [1.1843e-01, 4.5811e-02, 8.3576e-01],
        [4.3709e-05, 9.9370e-01, 6.2586e-03]])

Predicted classes:
tensor([0, 1, 2, 1])

Correct classes:
tensor([0, 1, 2, 1])

------------------------------

VALIDATION SET
Raw logits:
tensor([[ 0.6782, -1.1450,  0.1603],
        [-0.3644, -2.7257,  1.1106],
        [-5.9799,  2.6600, -1.4966]])

Probabilities:
tensor([[5.6906e-01, 9.1903e-02, 3.3904e-01],
        [1.8298e-01, 1.7253e-02, 7.9977e-01],
        [1.7416e-04, 9.8441e-01, 1.5418e-02]])

Predicted classes:
tensor([0, 2, 1])

Correct classes:
tensor([0, 2, 1])


### Why can `argmax()` be applied directly to logits?

Suppose the logits are:

```text
[2.0, 1.0, 0.1]
```

Softmax might produce approximately:

```text
[0.659, 0.242, 0.099]
```

The largest logit and the largest Softmax probability occur at the **same class index**.

Therefore:

```python
predictions = logits.argmax(dim=1)
```

is enough to determine the predicted class.


# Final rules to remember

## During training

Use the **training set** to calculate the loss used for backpropagation:

```python
training_logits = model(X)
training_loss = loss_function(training_logits, Y)

training_loss.backward()
optimizer.step()
```

## During validation

Use the validation set only to measure performance:

```python
model.eval()

with torch.no_grad():
    validation_logits = model(X_validation)
    validation_loss = loss_function(
        validation_logits,
        Y_validation
    )
```

Do **not** call `backward()` or `optimizer.step()` using the validation loss.

## When displaying probabilities

```python
logits = model(X)
probabilities = torch.softmax(logits, dim=1)
```

> **Do not put Softmax inside `forward()` when using `nn.CrossEntropyLoss()`. Let `forward()` return raw logits and pass those logits directly to the loss function.**
